# 📈 Real-Time Financial Market Dashboard
**Tech Stack:** Python | yfinance | Pandas | Plotly | Statistics

### What this project does:
- Pulls live stock data for 15 tickers using yfinance
- Calculates KPIs: 50-day MA, RSI, Bollinger Bands, Sharpe Ratio
- Detects anomalies (returns > 2 standard deviations)
- Sector correlation heatmap
- Monte Carlo price forecasting
- Saves all charts as HTML files (for GitHub portfolio)

In [ ]:
# ─── STEP 0: Install required libraries (run once) ───────────────────────────
import subprocess, sys

required = ['yfinance', 'plotly', 'pandas', 'numpy', 'scipy', 'kaleido']
for pkg in required:
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'{pkg} installed ✓')

print('\n✅ All libraries ready!')

In [ ]:
# ─── STEP 1: Imports ──────────────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings, os
warnings.filterwarnings('ignore')

os.makedirs('financial_dashboard_output', exist_ok=True)
print('📁 Output folder: financial_dashboard_output/')

In [ ]:
# ─── STEP 2: Download Stock Data ─────────────────────────────────────────────
TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META',   # Tech
    'JPM',  'BAC',  'GS',                        # Finance
    'JNJ',  'PFE',                               # Healthcare
    'XOM',  'CVX',                               # Energy
    'TSLA', 'NVDA', 'AMD'                        # Growth
]

PERIOD   = '1y'   # 1 year of data
INTERVAL = '1d'   # Daily bars

print(f'⬇️  Downloading data for {len(TICKERS)} tickers...')
try:
    raw = yf.download(TICKERS, period=PERIOD, interval=INTERVAL,
                      auto_adjust=True, progress=False)
    # Handle MultiIndex vs single-level columns
    if isinstance(raw.columns, pd.MultiIndex):
        prices = raw['Close'].copy()
    else:
        prices = raw[['Close']].copy()
    prices.dropna(how='all', inplace=True)
    print(f'✅ Downloaded {len(prices)} trading days × {prices.shape[1]} tickers')
    print(prices.tail(3))
except Exception as e:
    print(f'❌ Download error: {e}')
    print('💡 Check your internet connection and try again.')
    raise

In [ ]:
# ─── STEP 3: KPI Calculation Functions ───────────────────────────────────────

def calc_ma(series, window=50):
    return series.rolling(window=window, min_periods=1).mean()

def calc_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period, min_periods=1).mean()
    loss  = (-delta.clip(upper=0)).rolling(period, min_periods=1).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def calc_bollinger(series, window=20, std_dev=2):
    ma    = series.rolling(window, min_periods=1).mean()
    sigma = series.rolling(window, min_periods=1).std()
    return ma + std_dev * sigma, ma, ma - std_dev * sigma

def calc_sharpe(returns, risk_free=0.05):
    excess  = returns - risk_free / 252
    if excess.std() == 0:
        return 0
    return round((excess.mean() / excess.std()) * np.sqrt(252), 4)

def detect_anomalies(returns, threshold=2):
    mu, sigma = returns.mean(), returns.std()
    return returns[np.abs(returns - mu) > threshold * sigma]

# Compute KPIs for all tickers
kpi_summary = []
for ticker in prices.columns:
    s  = prices[ticker].dropna()
    if len(s) < 20:
        continue
    ret = s.pct_change().dropna()
    kpi_summary.append({
        'Ticker'       : ticker,
        'Last_Price'   : round(s.iloc[-1], 2),
        'MA_50'        : round(calc_ma(s).iloc[-1], 2),
        'RSI'          : round(calc_rsi(s).iloc[-1], 2),
        'Sharpe_Ratio' : calc_sharpe(ret),
        'Daily_Ret_%'  : round(ret.iloc[-1] * 100, 3),
        'Anomaly_Days' : len(detect_anomalies(ret)),
        'Volatility_%' : round(ret.std() * np.sqrt(252) * 100, 2)
    })

kpi_df = pd.DataFrame(kpi_summary)
print('📊 KPI Summary Table:')
print(kpi_df.to_string(index=False))

In [ ]:
# ─── STEP 4: Interactive KPI Table ───────────────────────────────────────────
fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=[f'<b>{c}</b>' for c in kpi_df.columns],
        fill_color='#1f2937', font=dict(color='white', size=12),
        align='center', height=35
    ),
    cells=dict(
        values=[kpi_df[c] for c in kpi_df.columns],
        fill_color=[['#f0fdf4' if i % 2 == 0 else 'white'
                     for i in range(len(kpi_df))]],
        align='center', height=28
    )
)])
fig_table.update_layout(title='📊 Financial KPI Summary — All Tickers',
                        height=500, margin=dict(l=10, r=10, t=50, b=10))
fig_table.show()
fig_table.write_html('financial_dashboard_output/01_kpi_table.html')
print('✅ Saved: 01_kpi_table.html')

In [ ]:
# ─── STEP 5: Bollinger Band + MA Chart (AAPL example) ────────────────────────
FOCUS = 'AAPL'
s     = prices[FOCUS].dropna()
bb_up, bb_mid, bb_low = calc_bollinger(s)
ma50  = calc_ma(s, 50)
ret   = s.pct_change().dropna()
anom  = detect_anomalies(ret)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    row_heights=[0.55, 0.25, 0.20],
                    subplot_titles=(f'{FOCUS} Price + Bollinger Bands',
                                    'RSI (14)', 'Daily Returns + Anomalies'))

# Price + Bollinger
fig.add_trace(go.Scatter(x=s.index, y=bb_up,  name='BB Upper',
              line=dict(color='red',   dash='dot'), opacity=0.6), row=1, col=1)
fig.add_trace(go.Scatter(x=s.index, y=bb_low, name='BB Lower',
              fill='tonexty', fillcolor='rgba(200,0,0,0.07)',
              line=dict(color='red',   dash='dot'), opacity=0.6), row=1, col=1)
fig.add_trace(go.Scatter(x=s.index, y=s,      name='Close',
              line=dict(color='#0ea5e9', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=s.index, y=ma50,   name='MA-50',
              line=dict(color='orange', dash='dash')), row=1, col=1)

# RSI
rsi = calc_rsi(s)
fig.add_trace(go.Scatter(x=rsi.index, y=rsi, name='RSI',
              line=dict(color='purple')), row=2, col=1)
fig.add_hline(y=70, line_dash='dot', line_color='red',   row=2, col=1)
fig.add_hline(y=30, line_dash='dot', line_color='green', row=2, col=1)

# Returns + Anomalies
fig.add_trace(go.Bar(x=ret.index, y=ret*100, name='Daily Return %',
              marker_color='#6366f1'), row=3, col=1)
if len(anom) > 0:
    fig.add_trace(go.Scatter(x=anom.index, y=anom*100, mode='markers',
                  name='Anomaly', marker=dict(color='red', size=8, symbol='x')),
                  row=3, col=1)

fig.update_layout(height=800, title=f'📈 {FOCUS} — Full Technical Analysis',
                  template='plotly_white', showlegend=True)
fig.show()
fig.write_html('financial_dashboard_output/02_bollinger_rsi.html')
print('✅ Saved: 02_bollinger_rsi.html')

In [ ]:
# ─── STEP 6: Sector Correlation Heatmap ──────────────────────────────────────
returns_all = prices.pct_change().dropna()
corr        = returns_all.corr()

fig_heat = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r',
                     zmin=-1, zmax=1, aspect='auto',
                     title='🔥 Correlation Heatmap — 15 Tickers (Daily Returns)')
fig_heat.update_layout(height=600, template='plotly_white')
fig_heat.show()
fig_heat.write_html('financial_dashboard_output/03_correlation_heatmap.html')
print('✅ Saved: 03_correlation_heatmap.html')

In [ ]:
# ─── STEP 7: Sharpe Ratio Bar Chart ──────────────────────────────────────────
sharpe_df = kpi_df.sort_values('Sharpe_Ratio', ascending=False)

fig_sharpe = px.bar(sharpe_df, x='Ticker', y='Sharpe_Ratio',
                    color='Sharpe_Ratio', color_continuous_scale='RdYlGn',
                    text='Sharpe_Ratio',
                    title='🏆 Sharpe Ratio by Ticker (Higher = Better Risk-Adjusted Return)')
fig_sharpe.update_traces(textposition='outside')
fig_sharpe.add_hline(y=1, line_dash='dash', line_color='navy',
                     annotation_text='Target Sharpe = 1')
fig_sharpe.update_layout(height=450, template='plotly_white')
fig_sharpe.show()
fig_sharpe.write_html('financial_dashboard_output/04_sharpe_ratio.html')
print('✅ Saved: 04_sharpe_ratio.html')

In [ ]:
# ─── STEP 8: Monte Carlo Simulation ──────────────────────────────────────────
MC_TICKER = 'AAPL'
N_SIMS    = 300
N_DAYS    = 90   # 3-month forecast

s_mc   = prices[MC_TICKER].dropna()
ret_mc = s_mc.pct_change().dropna()
mu     = ret_mc.mean()
sigma  = ret_mc.std()
S0     = s_mc.iloc[-1]

np.random.seed(42)
paths = np.zeros((N_DAYS, N_SIMS))
for i in range(N_SIMS):
    shocks     = np.random.normal(mu, sigma, N_DAYS)
    paths[:, i] = S0 * np.cumprod(1 + shocks)

future_idx = pd.bdate_range(s_mc.index[-1], periods=N_DAYS + 1)[1:]

fig_mc = go.Figure()
for i in range(min(N_SIMS, 100)):   # plot first 100 paths only
    fig_mc.add_trace(go.Scatter(x=future_idx, y=paths[:, i],
                     mode='lines', line=dict(width=0.5, color='rgba(99,102,241,0.2)'),
                     showlegend=False))

# Median & percentiles
p5  = np.percentile(paths, 5,  axis=1)
p50 = np.percentile(paths, 50, axis=1)
p95 = np.percentile(paths, 95, axis=1)
fig_mc.add_trace(go.Scatter(x=future_idx, y=p50, name='Median',
                            line=dict(color='orange', width=2.5)))
fig_mc.add_trace(go.Scatter(x=future_idx, y=p95, name='95th pct',
                            line=dict(color='green', dash='dash')))
fig_mc.add_trace(go.Scatter(x=future_idx, y=p5,  name='5th pct',
                            line=dict(color='red',   dash='dash')))

fig_mc.update_layout(
    title=f'🎲 Monte Carlo Simulation — {MC_TICKER} 90-Day Price Forecast ({N_SIMS} paths)',
    xaxis_title='Date', yaxis_title='Price (USD)',
    template='plotly_white', height=500
)
fig_mc.show()
fig_mc.write_html('financial_dashboard_output/05_monte_carlo.html')

# Summary stats
print(f'\n📌 {MC_TICKER} 90-Day Monte Carlo Summary:')
print(f'  Current Price : ${S0:.2f}')
print(f'  Median (50th) : ${p50[-1]:.2f}  ({(p50[-1]/S0-1)*100:+.2f}%)')
print(f'  Bull  (95th)  : ${p95[-1]:.2f}  ({(p95[-1]/S0-1)*100:+.2f}%)')
print(f'  Bear  (5th)   : ${p5[-1]:.2f}  ({(p5[-1]/S0-1)*100:+.2f}%)')
print('✅ Saved: 05_monte_carlo.html')

In [ ]:
# ─── STEP 9: Export KPI to CSV ────────────────────────────────────────────────
kpi_df.to_csv('financial_dashboard_output/kpi_summary.csv', index=False)
print('✅ KPI table saved: kpi_summary.csv')

print('\n' + '='*55)
print('🎉  PROJECT 1 COMPLETE!')
print('='*55)
print('📂 Output files saved in: financial_dashboard_output/')
print('   01_kpi_table.html')
print('   02_bollinger_rsi.html')
print('   03_correlation_heatmap.html')
print('   04_sharpe_ratio.html')
print('   05_monte_carlo.html')
print('   kpi_summary.csv')
print('\n📌 CV Line:')
print('   Developed a real-time stock analytics app (Python) ingesting')
print('   data via REST APIs for 15 tickers. Implemented Bollinger Bands,')
print('   RSI, Sharpe ratio & anomaly detection with Monte Carlo simulation.')